# Label Scenario 5 


In [3]:
#
# Cell 1: Libraries laden
#

import pandas as pd
import numpy as np
import wfdb
import ast

In [5]:
#
# Cell 2:Path for input data and definition of sampling rate
#

#path = 'C:/Users/Buchh/OneDrive/Leuphana_Data_Science/Masterarbeit/ECG/Python/Daten_einlesen/test_data/'
path = 'E:/MA_Daten_2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
sampling_rate=100


In [7]:
#
# Cell 3: Definition of functions
#

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

def aggregate_form_rhythm(y_dic): #BuK
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].form_rhythm)
    return list(set(tmp))


In [9]:
#
# Cell 3a: Specify the directory name for loading of data
#

from pathlib import Path

directory_path = Path("D:/ECG/X_zero")

In [11]:
#
# Cell 4:Loading stored raw data, if available. 
#

Y = pd.read_pickle(directory_path/'Y_data.pkl')

In [13]:
#
# Cell 5: FORM AND RHYTHM LABELS
# Load scp_statements.csv for form and rhythm aggregation (BuK)
#

agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[(agg_df.form == 1) | (agg_df.rhythm == 1)]
agg_df['form_rhythm'] = agg_df.index

In [15]:
#
#Cell 6: Apply form and rhythm class, BuK
#

Y['label'] = Y.scp_codes.apply(aggregate_form_rhythm)
display(Y)



,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,label
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[LVOLT, SR]"
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[SBRAD]
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[SR]
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[SR]
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[SR]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr,"[NDT, PVC, VCLVH, STACH]"
21795,21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,"[ABQRS, SR]"
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr,[SR]
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr,[SR]


In [17]:
#
#Cell 7: Make a copy from Y for further manipulation
#

Y_copy = Y[['ecg_id','patient_id','strat_fold','label']].copy() 
Y_copy

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[LVOLT, SR]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
21794,21833,17180.0,7,"[NDT, PVC, VCLVH, STACH]"
21795,21834,20703.0,4,"[ABQRS, SR]"
21796,21835,19311.0,2,[SR]
21797,21836,8873.0,8,[SR]


## Y-data manipulation for augmentation starts 

In [17]:
#
# Cell 7a: 
#

from pathlib import Path
directory_path_augm = Path("d:/ECG/X_augmented")

In [19]:
#
# Cell 7b: loads dicitionary with augmented X-xata
#

import pickle

with open(directory_path_augm/'X_augmented_accomp.pkl', 'rb') as f:
    X_augmented_accomp = pickle.load(f) # deserialize using load()
f.close()  

X_augmented_accomp

{'# of additional X-Data concatenated': 2}

In [21]:
#
# Cell 7c: select a certain row for later control purposes:
#

Y_21794 = Y_copy[Y_copy.index==21794].copy()
Y_21794

,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[NDT, VCLVH, STACH, PVC]"


In [23]:
#
# Cell 7d: concatenates the df Y_copy according to the value of  
# '# of additional X-Data concatenated':
#

Y_add=Y_copy
for i in range(0, X_augmented_accomp.get('# of additional X-Data concatenated')):
    Y_copy= pd.concat([Y_copy,Y_add], axis=0, ignore_index= True).copy()

Y_copy    

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[LVOLT, SR]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
65392,21833,17180.0,7,"[NDT, VCLVH, STACH, PVC]"
65393,21834,20703.0,4,"[ABQRS, SR]"
65394,21835,19311.0,2,[SR]
65395,21836,8873.0,8,[SR]


In [27]:
#
# Cell 7e: compare two df extracts for control of correct index assignment  
# 21749 + 21799 = 43593
#

Y_43593 = Y_copy[Y_copy.index==43593].copy()
display(Y_43593)
display(Y_21794)

,ecg_id,patient_id,strat_fold,label
43593,21833,17180.0,7,"[NDT, VCLVH, STACH, PVC]"


,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[NDT, VCLVH, STACH, PVC]"


In [29]:
#
# Cell 7f: 
#

Y_65392 = Y_copy[Y_copy.index==65392].copy()
display(Y_65392)

,ecg_id,patient_id,strat_fold,label
65392,21833,17180.0,7,"[NDT, VCLVH, STACH, PVC]"


## Y-data manipulation for augmentation ends
## The remaining cells are indentical with the cells of the modul for not augmented data   

In [19]:
#
# Cell 8: in all combinations of labels with "SR" remove "SR" 
# clean list of labels is now stored in the column "codes"
#

Y_copy["codes"] = Y_copy["label"].apply(
    lambda lst: ["SR"] if len(lst) == 1 and lst[0] == "SR"
    else sorted([c for c in lst if c != "SR"])
)
Y_copy

,ecg_id,patient_id,strat_fold,label,codes
0,1,15709.0,3,"[LVOLT, SR]",[LVOLT]
1,2,13243.0,2,[SBRAD],[SBRAD]
2,3,20372.0,5,[SR],[SR]
3,4,17014.0,3,[SR],[SR]
4,5,17448.0,4,[SR],[SR]
...,...,...,...,...,...
21794,21833,17180.0,7,"[NDT, PVC, VCLVH, STACH]","[NDT, PVC, STACH, VCLVH]"
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS]
21796,21835,19311.0,2,[SR],[SR]
21797,21836,8873.0,8,[SR],[SR]


In [21]:
#
# Cell 9: count the items in the lists in column "codes" and write to "len" column 
#

Y_copy["len"] = Y_copy["codes"].apply(
    lambda lst: len(lst)
)
Y_copy

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21794,21833,17180.0,7,"[NDT, PVC, VCLVH, STACH]","[NDT, PVC, STACH, VCLVH]",4
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


In [23]:
#
# Cell 10: Removes from "Y_copy" all rows resp. ECGs with empty codes
#

Y_copy=Y_copy[Y_copy['len']>0].copy() 
Y_copy

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21794,21833,17180.0,7,"[NDT, PVC, VCLVH, STACH]","[NDT, PVC, STACH, VCLVH]",4
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


In [25]:
#
# Cell 11: - convert list element of column "codes" into objects
#

Y_copy['codes'] = Y_copy['codes'].apply(lambda x: ", ".join(x)).copy()
Y_copy

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
...,...,...,...,...,...,...
21794,21833,17180.0,7,"[NDT, PVC, VCLVH, STACH]","NDT, PVC, STACH, VCLVH",4
21795,21834,20703.0,4,"[ABQRS, SR]",ABQRS,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


In [27]:
#
# Cell 12:
#

all_other_allone = Y_copy[~Y_copy['codes'].str.contains('AFIB|ABQRS|SR')].copy()
all_other_allone
#Y_copy[~Y_copy['codes'].str.contains(r'^(?=.*AFIB)(?=.*ABQRS)')]

#df['good_or_east'] = df['team'].str.contains('|'.join(['Good', 'East']))

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
10,11,11243.0,5,[SARRH],SARRH,1
11,12,11031.0,8,[SBRAD],SBRAD,1
14,15,13375.0,3,[SARRH],SARRH,1
...,...,...,...,...,...,...
21764,21803,12842.0,2,[SARRH],SARRH,1
21767,21806,20136.0,5,"[NDT, SR]",NDT,1
21777,21816,15103.0,3,"[NDT, SR]",NDT,1
21778,21817,18354.0,9,[PACE],PACE,1


In [29]:
#
# Cell 12a:Inserts a new colum in the dataframe with the simplified labels
# resp. codes
#

all_other_allone.loc[:, 'codes_2'] = 'all_other_allone'
all_other_allone

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1,all_other_allone
1,2,13243.0,2,[SBRAD],SBRAD,1,all_other_allone
10,11,11243.0,5,[SARRH],SARRH,1,all_other_allone
11,12,11031.0,8,[SBRAD],SBRAD,1,all_other_allone
14,15,13375.0,3,[SARRH],SARRH,1,all_other_allone
...,...,...,...,...,...,...,...
21764,21803,12842.0,2,[SARRH],SARRH,1,all_other_allone
21767,21806,20136.0,5,"[NDT, SR]",NDT,1,all_other_allone
21777,21816,15103.0,3,"[NDT, SR]",NDT,1,all_other_allone
21778,21817,18354.0,9,[PACE],PACE,1,all_other_allone


In [31]:
#
# Cell 13: Separates the ECGs with SR label not wnated by now (but later).
#

Y_SR_not_wanted = Y_copy[Y_copy['codes'].str.contains(r'^(?=.*SR)')].copy()
Y_SR_not_wanted

,ecg_id,patient_id,strat_fold,label,codes,len
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
5,6,19005.0,4,[SR],SR,1
6,7,16193.0,7,[SR],SR,1
...,...,...,...,...,...,...
21792,21831,11905.0,9,[SR],SR,1
21793,21832,7954.0,7,[SR],SR,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


In [33]:
#
# Cell 14: Substract the not wanted SR ECGs from Y_copy (all ECGs)
#

Y_ABQRS_AFIB_all_other_df = Y_copy[~Y_copy.index.isin(Y_SR_not_wanted.index)]
Y_ABQRS_AFIB_all_other_df

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1
10,11,11243.0,5,[SARRH],SARRH,1
11,12,11031.0,8,[SBRAD],SBRAD,1
...,...,...,...,...,...,...
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2
21789,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2
21790,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1
21794,21833,17180.0,7,"[NDT, PVC, VCLVH, STACH]","NDT, PVC, STACH, VCLVH",4


In [35]:
#
# Cell 15: 
#

ABQRS_AFIB_plus_all_other = Y_ABQRS_AFIB_all_other_df[Y_ABQRS_AFIB_all_other_df['codes'].str.contains(r'^(?=.*AFIB)(?=.*ABQRS)')].copy()
ABQRS_AFIB_plus_all_other

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3
1888,1901,16011.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
2433,2446,12655.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2
3207,3222,18571.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2
3226,3241,20550.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
...,...,...,...,...,...,...
21703,21742,13857.0,8,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
21718,21757,13857.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2
21724,21763,13857.0,8,"[VCLVH, ABQRS, AFIB]","ABQRS, AFIB, VCLVH",3
21726,21765,15902.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2


In [37]:
#
# Cell 16. filter all instaces with one label
#

Y_label_len_one_df = Y_copy[Y_copy["len"]==1].copy()
Y_label_len_one_df[90:120]

,ecg_id,patient_id,strat_fold,label,codes,len
95,96,21698.0,1,[SR],SR,1
96,97,12761.0,6,[SR],SR,1
97,98,10804.0,8,[SARRH],SARRH,1
98,99,18012.0,4,[SR],SR,1
99,100,21404.0,10,[SARRH],SARRH,1
100,101,9086.0,4,[SR],SR,1
101,102,9887.0,1,[SR],SR,1
103,104,19738.0,4,[SR],SR,1
104,105,18974.0,3,"[ABQRS, SR]",ABQRS,1
105,106,16066.0,5,"[ABQRS, SR]",ABQRS,1


In [39]:
#
# Cell 17: filters all instaces with two label
#

Y_label_len_two_df = Y_copy[Y_copy["len"]==2].copy()
Y_label_len_two_df

,ecg_id,patient_id,strat_fold,label,codes,len
19,20,13619.0,9,"[AFLT, ABQRS]","ABQRS, AFLT",2
25,26,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
27,28,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2
162,170,15691.0,9,"[NST_, PVC, SR]","NST_, PVC",2
...,...,...,...,...,...,...
21773,21812,20789.0,10,"[SVARR, ABQRS]","ABQRS, SVARR",2
21779,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2
21787,21826,9178.0,10,"[SARRH, ABQRS]","ABQRS, SARRH",2
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2


In [41]:
#
# Cell 18: filters all instaces with two and more label
#

Y_label_len_greater_one_df = Y_copy[Y_copy["len"]>1].copy()
Y_label_len_greater_one_df

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3
19,20,13619.0,9,"[AFLT, ABQRS]","ABQRS, AFLT",2
25,26,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
27,28,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
38,39,13619.0,9,"[NST_, DIG, ABQRS, LNGQT, SR]","ABQRS, DIG, LNGQT, NST_",4
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2
21787,21826,9178.0,10,"[SARRH, ABQRS]","ABQRS, SARRH",2
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2
21789,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2


In [43]:
#
# Cell 19: TWO LABEL - filters all instances with binary label combinations ABQRS and AFIB
#

ABQRS_AFIB = Y_label_len_two_df[Y_label_len_two_df['codes'].str.contains(r'^(?=.*AFIB)(?=.*ABQRS)')].copy()
ABQRS_AFIB

,ecg_id,patient_id,strat_fold,label,codes,len
1888,1901,16011.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
2433,2446,12655.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2
3207,3222,18571.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2
3226,3241,20550.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
3691,3706,15521.0,3,"[ABQRS, AFIB]","ABQRS, AFIB",2
...,...,...,...,...,...,...
21489,21528,13246.0,4,"[ABQRS, AFIB]","ABQRS, AFIB",2
21661,21700,12791.0,5,"[ABQRS, AFIB]","ABQRS, AFIB",2
21718,21757,13857.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2
21726,21765,15902.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2


In [45]:
#
# Cell 19a: Inserts a new colum in the dataframe with the simplified labels
# resp. codes
#

ABQRS_AFIB.loc[:, 'codes_2'] = 'ABQRS_AFIB'
ABQRS_AFIB

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
1888,1901,16011.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
2433,2446,12655.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
3207,3222,18571.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
3226,3241,20550.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
3691,3706,15521.0,3,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
...,...,...,...,...,...,...,...
21489,21528,13246.0,4,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
21661,21700,12791.0,5,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
21718,21757,13857.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
21726,21765,15902.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB


# Following Cell build the input data

"ABQRS+AFIB" (2 of 8)

In [47]:
#
# Cell 20: TWO LABEL - Puts filtered ECGs in dictionary
#

ABQRS_AFIB_dfs = {
    'ABQRS_AFIB': ABQRS_AFIB} 
print(ABQRS_AFIB_dfs) 

{'ABQRS_AFIB':        ecg_id  patient_id  strat_fold          label        codes  len  \
1888     1901     16011.0          10  [ABQRS, AFIB]  ABQRS, AFIB    2   
2433     2446     12655.0           8  [ABQRS, AFIB]  ABQRS, AFIB    2   
3207     3222     18571.0           9  [ABQRS, AFIB]  ABQRS, AFIB    2   
3226     3241     20550.0          10  [ABQRS, AFIB]  ABQRS, AFIB    2   
3691     3706     15521.0           3  [ABQRS, AFIB]  ABQRS, AFIB    2   
...       ...         ...         ...            ...          ...  ...   
21489   21528     13246.0           4  [ABQRS, AFIB]  ABQRS, AFIB    2   
21661   21700     12791.0           5  [ABQRS, AFIB]  ABQRS, AFIB    2   
21718   21757     13857.0           8  [ABQRS, AFIB]  ABQRS, AFIB    2   
21726   21765     15902.0           9  [ABQRS, AFIB]  ABQRS, AFIB    2   
21757   21796     17746.0          10  [ABQRS, AFIB]  ABQRS, AFIB    2   

          codes_2  
1888   ABQRS_AFIB  
2433   ABQRS_AFIB  
3207   ABQRS_AFIB  
3226   ABQRS_AFI

In [49]:
#
# Cell 21: Remove all binary label combinations of AFIB and ABQRS label from
# the set of AFIB, ABQRS and all_other label combinations to get a intersection
# of all three sets (AFIB, ABQRS and all_other label)
#

Y_intersection_all =ABQRS_AFIB_plus_all_other[~ABQRS_AFIB_plus_all_other.index.isin(ABQRS_AFIB.index)].copy()
Y_intersection_all

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3
3439,3454,11760.0,10,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
3535,3550,19766.0,9,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
5119,5139,21190.0,9,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
5144,5164,12949.0,7,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
...,...,...,...,...,...,...
21324,21363,13640.0,10,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
21608,21647,10481.0,1,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
21616,21655,14805.0,7,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
21703,21742,13857.0,8,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3


In [51]:
#
# Cell 21a:Inserts a new colum in the dataframe with the simplified labels
# resp. codes

Y_intersection_all.loc[:, 'codes_2'] = 'intersection_all'
Y_intersection_all

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3,intersection_all
3439,3454,11760.0,10,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
3535,3550,19766.0,9,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
5119,5139,21190.0,9,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
5144,5164,12949.0,7,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
...,...,...,...,...,...,...,...
21324,21363,13640.0,10,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
21608,21647,10481.0,1,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
21616,21655,14805.0,7,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
21703,21742,13857.0,8,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all


In [53]:
#
# Cell 22: selects the scp-codes of interest
#

codes_list=['SR', 'ABQRS', 'AFIB']

In [55]:
#
# Cell 23: makes a dictionary from list codes_list
#

dfs= {}
for item in codes_list:
    #dfs['Y_'+item] = item
    dfs[item] = item
dfs

{'SR': 'SR', 'ABQRS': 'ABQRS', 'AFIB': 'AFIB'}

# Following Cell build the input data 

"SR", 

"ABQRS", 

"AFIB" (5 of 8) 

In [57]:
#
# Cell 24: selects all ECGs with single label 
# according code_list and collects them in dictionary individual_dfs
#

individual_dfs = {
    name: Y_label_len_one_df[Y_label_len_one_df['codes'] == code].copy()
    for name, code in dfs.items()
}

individual_dfs['ABQRS']

,ecg_id,patient_id,strat_fold,label,codes,len
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1
61,62,21273.0,9,"[ABQRS, SR]",ABQRS,1
62,63,15265.0,10,"[ABQRS, SR]",ABQRS,1
71,72,14060.0,9,"[ABQRS, SR]",ABQRS,1
72,73,19987.0,5,"[ABQRS, SR]",ABQRS,1
...,...,...,...,...,...,...
21772,21811,12934.0,7,"[ABQRS, SR]",ABQRS,1
21776,21815,14433.0,3,"[ABQRS, SR]",ABQRS,1
21785,21824,19977.0,8,"[ABQRS, SR]",ABQRS,1
21790,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1


In [59]:
#
# Cell 24a: Inserts a new colum in the dataframes with the simplified labels
# resp. codes
#

for name, df in individual_dfs.items():
    df = df.copy()
    df.loc[:, 'codes_2'] = name
    individual_dfs[name] = df

individual_dfs['ABQRS']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1,ABQRS
61,62,21273.0,9,"[ABQRS, SR]",ABQRS,1,ABQRS
62,63,15265.0,10,"[ABQRS, SR]",ABQRS,1,ABQRS
71,72,14060.0,9,"[ABQRS, SR]",ABQRS,1,ABQRS
72,73,19987.0,5,"[ABQRS, SR]",ABQRS,1,ABQRS
...,...,...,...,...,...,...,...
21772,21811,12934.0,7,"[ABQRS, SR]",ABQRS,1,ABQRS
21776,21815,14433.0,3,"[ABQRS, SR]",ABQRS,1,ABQRS
21785,21824,19977.0,8,"[ABQRS, SR]",ABQRS,1,ABQRS
21790,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1,ABQRS


In [61]:
#
# Cell 25: ALL OTHER - substract Y_label_len_one_short_df from Y_copy
#

Y_all_other_plus_df = Y_copy[~Y_copy.index.isin(Y_label_len_one_df.index)]
Y_all_other_plus_df

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3
19,20,13619.0,9,"[AFLT, ABQRS]","ABQRS, AFLT",2
25,26,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
27,28,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
38,39,13619.0,9,"[NST_, DIG, ABQRS, LNGQT, SR]","ABQRS, DIG, LNGQT, NST_",4
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2
21787,21826,9178.0,10,"[SARRH, ABQRS]","ABQRS, SARRH",2
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2
21789,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2


In [63]:
#
# Cell 26: Select all ECGs with both "ABQRS" and "AFIB"
#

ABQRS_AFIB_not_wanted = Y_all_other_plus_df[Y_all_other_plus_df['codes'].str.contains(r'^(?=.*AFIB)(?=.*ABQRS)')].copy()
ABQRS_AFIB_not_wanted

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3
1888,1901,16011.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
2433,2446,12655.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2
3207,3222,18571.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2
3226,3241,20550.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
...,...,...,...,...,...,...
21703,21742,13857.0,8,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3
21718,21757,13857.0,8,"[ABQRS, AFIB]","ABQRS, AFIB",2
21724,21763,13857.0,8,"[VCLVH, ABQRS, AFIB]","ABQRS, AFIB, VCLVH",3
21726,21765,15902.0,9,"[ABQRS, AFIB]","ABQRS, AFIB",2


In [65]:
#
# Cell 27: substracts "ABQRS_AFIB_not_wanted" from "Y_copy"
#

Y_all_other_plus_df = Y_label_len_greater_one_df[~Y_label_len_greater_one_df.index.isin(ABQRS_AFIB_not_wanted.index)]
Y_all_other_plus_df

,ecg_id,patient_id,strat_fold,label,codes,len
19,20,13619.0,9,"[AFLT, ABQRS]","ABQRS, AFLT",2
25,26,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
27,28,13619.0,9,"[NST_, DIG, SR]","DIG, NST_",2
38,39,13619.0,9,"[NST_, DIG, ABQRS, LNGQT, SR]","ABQRS, DIG, LNGQT, NST_",4
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2
21787,21826,9178.0,10,"[SARRH, ABQRS]","ABQRS, SARRH",2
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2
21789,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2


# The following Cell builds  the input data 

ABQRS + all_other, 

AFIB + all_other, 

SR_all_other (empty) (7 of 8) 

In [67]:
#
# Cell 28: selects all ECGs with two or more label 
# of those one label is one of the labels in code_list 
#

individual_plus_dfs = {
    name+'_all_other': Y_all_other_plus_df[Y_all_other_plus_df['codes'].str.contains(code)].copy()
    for name, code in dfs.items()
}

#df = df[df['Credit-Rating'].str.contains('Fair')]


individual_plus_dfs['ABQRS_all_other']

,ecg_id,patient_id,strat_fold,label,codes,len
19,20,13619.0,9,"[AFLT, ABQRS]","ABQRS, AFLT",2
38,39,13619.0,9,"[NST_, DIG, ABQRS, LNGQT, SR]","ABQRS, DIG, LNGQT, NST_",4
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2
232,240,9601.0,10,"[SBRAD, ABQRS]","ABQRS, SBRAD",2
788,801,15493.0,3,"[NST_, SBRAD, ABQRS]","ABQRS, NST_, SBRAD",3
...,...,...,...,...,...,...
21773,21812,20789.0,10,"[SVARR, ABQRS]","ABQRS, SVARR",2
21779,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2
21787,21826,9178.0,10,"[SARRH, ABQRS]","ABQRS, SARRH",2
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2


In [69]:
#
# Cell 28a: remove the empty dataframe "SR_all_other" from dicitionary 
#

del individual_plus_dfs['SR_all_other']

In [71]:
#
# Cell 28a: Inserts a new colum in the dataframes with the simplified labels
# resp. codes
#

for name, df in individual_plus_dfs.items():
    df = df.copy()
    df.loc[:, 'codes_2'] = name
    individual_plus_dfs[name] = df

individual_plus_dfs['ABQRS_all_other']


,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
19,20,13619.0,9,"[AFLT, ABQRS]","ABQRS, AFLT",2,ABQRS_all_other
38,39,13619.0,9,"[NST_, DIG, ABQRS, LNGQT, SR]","ABQRS, DIG, LNGQT, NST_",4,ABQRS_all_other
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2,ABQRS_all_other
232,240,9601.0,10,"[SBRAD, ABQRS]","ABQRS, SBRAD",2,ABQRS_all_other
788,801,15493.0,3,"[NST_, SBRAD, ABQRS]","ABQRS, NST_, SBRAD",3,ABQRS_all_other
...,...,...,...,...,...,...,...
21773,21812,20789.0,10,"[SVARR, ABQRS]","ABQRS, SVARR",2,ABQRS_all_other
21779,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2,ABQRS_all_other
21787,21826,9178.0,10,"[SARRH, ABQRS]","ABQRS, SARRH",2,ABQRS_all_other
21788,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2,ABQRS_all_other


In [73]:
#
# Cell 29: combine the two dictionaries "individual_plus_dfs" and "individual_dfs"
#

Y_sets_all_1 = individual_plus_dfs | individual_dfs

Y_sets_all_1['AFIB_all_other']


,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
274,282,437.0,5,"[PVC, AFIB]","AFIB, PVC",2,AFIB_all_other
299,307,6348.0,1,"[INVT, AFIB, STD_]","AFIB, INVT, STD_",3,AFIB_all_other
310,318,2034.0,10,"[NDT, DIG, AFIB, STD_]","AFIB, DIG, NDT, STD_",4,AFIB_all_other
313,321,4188.0,7,"[NST_, AFIB]","AFIB, NST_",2,AFIB_all_other
329,337,3498.0,8,"[NDT, LOWT, AFIB, STD_]","AFIB, LOWT, NDT, STD_",4,AFIB_all_other
...,...,...,...,...,...,...,...
21694,21733,9580.0,4,"[NDT, AFIB]","AFIB, NDT",2,AFIB_all_other
21695,21734,18491.0,8,"[PVC, AFIB]","AFIB, PVC",2,AFIB_all_other
21700,21739,14330.0,8,"[NST_, AFIB]","AFIB, NST_",2,AFIB_all_other
21740,21779,10590.0,5,"[NST_, AFIB]","AFIB, NST_",2,AFIB_all_other


In [75]:
#
# Cell 30: Add the remaining and still missing df to the dictionary "Y_sets_all_1"
#

Y_sets_all_1.update({'all_other_allone':all_other_allone,'ABQRS_AFIB':ABQRS_AFIB, 
                     'Y_intersection_all':Y_intersection_all})

Y_sets_all_1['Y_intersection_all']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
16,17,13619.0,9,"[AFLT, ABQRS, AFIB]","ABQRS, AFIB, AFLT",3,intersection_all
3439,3454,11760.0,10,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
3535,3550,19766.0,9,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
5119,5139,21190.0,9,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
5144,5164,12949.0,7,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
...,...,...,...,...,...,...,...
21324,21363,13640.0,10,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
21608,21647,10481.0,1,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
21616,21655,14805.0,7,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all
21703,21742,13857.0,8,"[PVC, ABQRS, AFIB]","ABQRS, AFIB, PVC",3,intersection_all


In [77]:
#
# Cell 31: Selects from the dataframe in Y_sets_all_1 all the instances
# with strat_fold == 10 an put them into separate dataframe
#

test_dfs = {name: df[df['strat_fold'] == 10].copy() for name, df in Y_sets_all_1.items()}
test_dfs['all_other_allone']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
75,76,21214.0,10,[SARRH],SARRH,1,all_other_allone
99,100,21404.0,10,[SARRH],SARRH,1,all_other_allone
115,116,19695.0,10,"[NDT, SR]",NDT,1,all_other_allone
121,122,19832.0,10,[SARRH],SARRH,1,all_other_allone
127,128,17735.0,10,[NDT],NDT,1,all_other_allone
...,...,...,...,...,...,...,...
20982,21021,14309.0,10,"[VCLVH, SR]",VCLVH,1,all_other_allone
21032,21071,14180.0,10,[PACE],PACE,1,all_other_allone
21286,21325,17347.0,10,[STACH],STACH,1,all_other_allone
21516,21555,15028.0,10,"[VCLVH, SR]",VCLVH,1,all_other_allone


In [79]:
#
# Cell 32: substracts for all classes the corresponding test instances from the sets.
# Generates the dictionary "residue_dfs" with all the residue df of the individual classes  
#

residue_dfs = {name: df[~df.index.isin(test_dfs[name].index)].copy() for name, df in Y_sets_all_1.items()}

residue_dfs['all_other_allone']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1,all_other_allone
1,2,13243.0,2,[SBRAD],SBRAD,1,all_other_allone
10,11,11243.0,5,[SARRH],SARRH,1,all_other_allone
11,12,11031.0,8,[SBRAD],SBRAD,1,all_other_allone
14,15,13375.0,3,[SARRH],SARRH,1,all_other_allone
...,...,...,...,...,...,...,...
21764,21803,12842.0,2,[SARRH],SARRH,1,all_other_allone
21767,21806,20136.0,5,"[NDT, SR]",NDT,1,all_other_allone
21777,21816,15103.0,3,"[NDT, SR]",NDT,1,all_other_allone
21778,21817,18354.0,9,[PACE],PACE,1,all_other_allone


In [104]:
#
# Cell 33: splits the df in the dictionary "residue_dfs" in training and validation sets.
#

training_dfs = {
    name: df.sample(frac=0.75, random_state=7).copy()
    for name, df in residue_dfs.items()
}


validation_dfs = {
    name.replace('_training', '_validation'): df.drop(training_dfs[name].index).copy()
    for name, df in residue_dfs.items()
}


display(training_dfs['all_other_allone'][0:5]) 
display(validation_dfs['all_other_allone'][0:5])

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
12006,12038,5526.0,1,"[SR, NT_]",NT_,1,all_other_allone
5518,5538,15698.0,9,"[LNGQT, VCLVH, SR]","LNGQT, VCLVH",2,all_other_allone
1458,1471,413.0,5,[SARRH],SARRH,1,all_other_allone
2660,2675,4483.0,6,"[SR, PAC, NT_]","NT_, PAC",2,all_other_allone
17092,17130,13903.0,9,"[NDT, SR]",NDT,1,all_other_allone


,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
1,2,13243.0,2,[SBRAD],SBRAD,1,all_other_allone
17,18,13619.0,9,[AFLT],AFLT,1,all_other_allone
36,37,9123.0,1,[SARRH],SARRH,1,all_other_allone
86,87,17293.0,7,"[NDT, SR]",NDT,1,all_other_allone
107,108,11810.0,8,[STACH],STACH,1,all_other_allone


In [106]:
#
# Cell 34: count the number of instances splits the df in 
# the dictionary "residue_dfs" in training and validation sets.
#

dfs_all= {1: test_dfs,
          2: training_dfs,
          3: validation_dfs
         }

summation=sum(df.shape[0] for df in dfs_all.values() for df in df.values())
print('Total sum of all instances in the different sets: ',summation)

Total sum of all instances in the different sets:  21297


In [108]:
#
# Cell 35: combine all test / validation / training sets of all sets (all classes built)
#

train_df = pd.concat(training_dfs.values(), axis=0).sample(frac=1, random_state=7)
val_df   = pd.concat(validation_dfs.values(), axis=0).sample(frac=1, random_state=7)
test_df  = pd.concat(test_dfs.values(), axis=0).sample(frac=1, random_state=7)
train_df

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
19703,19742,9694.0,5,[SR],SR,1,SR
9489,9513,17872.0,3,[SR],SR,1,SR
21798,21837,11744.0,9,[SR],SR,1,SR
3466,3481,418.0,1,"[SR, NT_]",NT_,1,all_other_allone
16261,16299,2784.0,7,"[PVC, SR]",PVC,1,all_other_allone
...,...,...,...,...,...,...,...
6957,6978,14632.0,4,[SR],SR,1,SR
6528,6549,12475.0,4,[SBRAD],SBRAD,1,all_other_allone
9881,9908,19456.0,9,"[NST_, AFIB]","AFIB, NST_",2,AFIB_all_other
20492,20531,10489.0,3,[AFIB],AFIB,1,AFIB


In [110]:
#
# Cell 36: 
#

Y_final_dfs = {'train_df' : train_df, 'val_df': val_df, 'test_df' : test_df}

## Following cell (37 to 42) are for preparing X data without augmentation 

In [94]:
#
#Cell 37: loads original X values (no augmentation) 
#

from pathlib import Path

X_directory_path = Path("D:/ECG/X_zero")
X = np.load(X_directory_path/'X_float32.npy', mmap_mode='r')

In [112]:
#
# Cell 38: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [98]:
#
# Cell 40: creates a folder for data storage.
#

# Specify the directory name
directory_path = Path("D:/ECG/XY_diff_class_4")

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_4' already exists.


In [114]:
#
# Cell 41: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays_3.npz', a = X_train, b = X_val, c = X_test)

In [116]:
#
# Cell 42: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data_3.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()

## The following cells must be executed if one wants to use the augmented datasets of X 

In [101]:
#
# Cell 43: path to read input data
#

X_directory_path = Path("D:/ECG/X_augmented_1000") # chose the right path beforhand
X = np.load(X_directory_path/'X_augmented.npz.npy', mmap_mode='r')

In [103]:
#
# Cell 44: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [105]:
#
# Cell 45: creates a folder for data storage.
#

# Specify the directory name
directory_path = Path("D:/ECG/XY_diff_class_4_aug_1000") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_4_aug_1000' already exists.


In [109]:
#
# Cell 46: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays.npz', a = X_train, b = X_val, c = X_test)

In [111]:
#
# Cell 47: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()

In [ ]:
# Cell 43: Restore X_augm_dfs with pickle 

with open(directory_path/'X_augmented.pkl', 'rb') as f:

    X_augm_dfs = pickle.load(f) # deserialize using load()
f.close()  

X_augm_dfs[K].shape